# OpenAI Assistants API를 활용한 서비스 구현 튜토리얼 (2025년 3월 기준)

본 튜토리얼에서는 OpenAI Assistants API를 활용하여 AI 어시스턴트 기반 서비스를 개발하는 방법을 단계별로 설명합니다. 2025년 3월 기준의 OpenAI 공식 API 문서를 기반으로 최신 기능과 모범 사례를 반영했으며, Python 예제 코드를 통해 실습 형태로 진행됩니다. 이 튜토리얼은 ChatGPT 등의 언어 모델 API 사용 경험이 있고, 새로운 Assistants API 기능에 관심이 있는 개발자를 대상으로 합니다.

## 1. OpenAI Assistants API 소개

OpenAI Assistants API는 OpenAI의 새로운 API로, 대화형 AI 어시스턴트를 보다 쉽게 구축하고 관리할 수 있도록 설계되었습니다. 기존의 Chat Completion API가 무상태(stateless) 방식이어서 매 요청마다 대화 기록을 모두 보내야 했다면, Assistants API는 상태를 유지(stateful) 하는 대화 스레드(Thread)를 제공하여 자동으로 대화 컨텍스트를 관리합니다. 또한 함수 호출, 코드 실행, 지식 검색 등의 도구(Tools) 를 어시스턴트에 병렬로 연결할 수 있어 훨씬 강력한 코파일럿(copilot) 형태의 서비스를 구축할 수 있습니다. 

왜 Assistants API를 사용해야 할까요? 몇 가지 주요 장점을 정리하면 다음과 같습니다:

- 대화 지속성 & 메모리: Assistants API는 대화 스레드에 메시지 내역을 지속적으로 저장하며, 컨텍스트 윈도우 제한 내에서 자동으로 요약/잘라내기를 수행합니다. 개발자는 별도의 상태 관리 로직 없이 장기간의 대화도 처리할 수 있습니다.

- 다양한 도구 통합: 어시스턴트가 코드 실행(Code Interpreter), 지식 검색(File Retrieval), 함수 호출(Function Calling) 등의 도구를 직접 사용하도록 구성할 수 있습니다. 이를 통해 모델의 한계를 넘어서는 작업(예: 데이터베이스 질의, 파일 처리, 외부 API 호출 등)을 자동화할 수 있습니다.

- 개인화된 지시어: 어시스턴트를 생성할 때 지시어(Instructions) 를 주입함으로써 해당 어시스턴트의 성격과 역할을 미리 정의할 수 있습니다. 예를 들어 “당신은 친절한 고객지원 봇입니다”와 같은 시스템 지시어로 어시스턴트의 톤과 도메인 지식을 설정할 수 있습니다.

- 모델 기능 향상: 2023년 11월 이후 공개된 GPT-3.5/4 모델 (예: gpt-3.5-turbo-1106, gpt-4-1106-preview)은 함수 호출과 같은 고급 기능을 지원하며 Assistants API와 연동됩니다. Assistants API를 사용하면 이러한 최신 모델 기능을 간편하게 활용할 수 있습니다.

- 개발 편의: OpenAI 플랫폼의 웹 인터페이스(플레이그라운드)뿐 아니라 API를 통해서도 동일한 어시스턴트를 생성/관리할 수 있습니다. 한 번 만든 어시스턴트를 재사용하거나 여러 쓰레드를 동시에 운영하는 것이 쉬워집니다. (예: 하나의 어시스턴트에 대해 다수의 사용자 대화 스레드를 관리)

요약하면, Assistants API는 기존 ChatGPT API의 상태 관리 진화판으로 볼 수 있습니다. 복잡한 대화 상태 관리, 외부 도구 통합, 문서 임베딩 등 많은 부분을 OpenAI 플랫폼이 맡아주므로, 개발자는 핵심 로직 구현에 집중할 수 있습니다. 이러한 이유로, 코파일럿 스타일의 챗봇이나 자동화 에이전트를 만든다면 Assistants API가 강력한 선택지가 됩니다.

>Note: Assistants API는 2025년 초 현재 베타(beta) 단계로, OpenAI가 지속적으로 기능을 개선하고 있습니다. 사용 전에 최신 문서를 확인하고, 베타 API인 만큼 일부 동작이나 요금 정책이 변경될 수 있음을 유의하세요.

## 2. API 키 설정 및 환경 변수 사용

OpenAI API를 사용하려면 API 키가 필요합니다. OpenAI 플랫폼의 API 키 관리 페이지에서 비밀 키를 생성할 수 있습니다. 생성된 키는 한 번만 표시되므로, 반드시 복사하여 안전한 곳에 저장하세요. 일반적으로 이 키를 소스 코드에 하드코딩하지 않고, 별도의 설정으로 관리하는 것이 좋습니다. **환경 변수(Environment Variable)**를 사용하면 API 키를 소스 코드에 노출하지 않고 관리할 수 있습니다. 개발 PC 또는 서버의 환경 변수 OPENAI_API_KEY에 키를 저장해 두면, OpenAI 라이브러리가 자동으로 이를 읽어 사용할 수 있습니다. Python 개발 환경에서는 python-dotenv 패키지를 활용해 .env 파일에 키를 저장하고 로드하는 방식이 편리합니다. 

다음은 API 키를 설정하고 로드하는 과정입니다:

1. python-dotenv 설치: 터미널에서 pip install python-dotenv 명령으로 설치합니다 (한번만 수행).

2. 환경 변수 파일 생성: 프로젝트 루트 디렉토리에 .env 파일을 만들고, 아래와 같이 OpenAI API 키를 입력합니다 (따옴표 없이 실제 키로 대체).

    ```
    OPENAI_API_KEY=sk-***********************
    ```
    
3. 코드에서 로드: Python 코드에서 python-dotenv를 이용해 .env를 로드하고, os.environ을 통해 키를 불러옵니다. OpenAI 공식 Python SDK에서는 환경 변수 OPENAI_API_KEY를 자동으로 인식하므로, 명시적으로 지정하지 않아도 동작합니다. 그래도 명확성을 위해 아래와 같이 작성할 수 있습니다:


In [2]:
# 6월 업그레이드 경고 무시
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

In [6]:
from dotenv import load_dotenv
from openai import OpenAI, OpenAIError
import os

# 1. OpenAi 클라이언트 생성
# print(os.getcwd())
load_dotenv(".env")
api_key = os.getenv("OPENAI_API_KEY")
print(api_key[:6], len(api_key))

client = OpenAI(api_key=api_key)

sk-pro 164


## 3. 기본적인 Assistant 생성 및 메시지 보내기

가장 먼저, **어시스턴트(Assistant)**를 하나 만들어보겠습니다. 어시스턴트는 대화를 수행하는 주체로, 사용자의 요청을 받아 처리하는 AI 에이전트라고 볼 수 있습니다. Assistants API에서 어시스턴트를 생성할 때는 어떤 모델을 사용할지, 어떤 지시어(Instructions) 를 가질지, 그리고 사용할 **툴(Tools)**이 무엇인지 등을 설정할 수 있습니다. 예를 들어 간단한 챗봇 어시스턴트를 만들어 사용자 질문에 답변해보겠습니다.


In [9]:
# 2. 어시스턴트 생성
assistant = client.beta.assistants.create(
    name="HelpBot",
    instructions="당신은 유능하고, 친절한 도움말 어시스턴트입니다. 사용자 질문에 20글자 이내로 친절하게 답변하세요",
    model="gpt-4.1-mini",
    # tools 매개변수를 생략하거나 빈리스트로 두면
)

In [15]:
# assistant
# print('llm에 요청시 필요한 assistant.id=', assistant.id)

위 코드에서는 client.beta.assistants.create 메서드를 사용해 새로운 어시스턴트를 생성했습니다. 주요 파라미터를 살펴보면:

- name: 어시스턴트의 이름을 지정합니다. 콘솔이나 리스트에서 구분하기 위한 용도이며, 모델 응답 내용에는 영향을 미치지 않습니다.

- instructions: 시스템 레벨의 지시어로, 해당 어시스턴트가 모든 대화에서 따르게 될 기본 규칙이나 역할을 정의합니다. 여기서는 "친절한 도움말 어시스턴트"라는 성격을 부여했습니다. (이 지시어는 기존 ChatGPT의 시스템 메시지와 유사한 역할입니다.)

- model: 어시스턴트가 사용할 언어 모델을 지정합니다. 최신 기능을 쓰려면 OpenAI가 Nov 2023 이후 출시한 모델(-1106가 붙은 모델명)을 권장합니다. 

- assistant 생성을 제공하는 모델
    GPT-4 계열:
    gpt-4 - 기본 GPT-4 모델
    gpt-4-turbo - 더 빠르고 효율적인 GPT-4
    gpt-4o - 최신 멀티모달 모델
    gpt-4o-mini - 경량화된 GPT-4o

    GPT-3.5 계열:
    gpt-3.5-turbo - 빠르고 효율적인 모델(추천)

    기타:
    dall-e-3 - 이미지 생성용
    whisper-1 - 음성 인식용
    tts-1, tts-1-hd - 텍스트 음성 변환용


- tools: 어시스턴트에 활성화할 도구 목록입니다. 기본적인 Q&A 챗봇에서는 특별한 툴이 필요 없으므로 비워두었습니다. (tools=[] 또는 생략) 나중에 코드 인터프리터나 함수 호출 등을 사용하게 될 경우 이 필드를 설정합니다.

어시스턴트가 성공적으로 생성되면 고유 ID (assistant.id)가 반환됩니다. 이 ID는 이후 대화 스레드에서 어떤 어시스턴트를 사용할지 지정할 때 필요하므로 저장해둡니다. (참고로, client.beta.assistants.list()를 호출하면 계정 내 모든 어시스턴트 목록을 확인할 수 있습니다.) 

이제 이 어시스턴트와 대화를 시작해보겠습니다. 대화를 하기 위해서는 **스레드(Thread)**를 생성해야 합니다. 스레드는 개별적인 대화 세션을 나타내며, 사용자와 어시스턴트 간의 메시지(Message) 목록을 포함합니다. 하나의 어시스턴트는 여러 개의 스레드를 가질 수 있는데, 예를 들어 같은 어시스턴트를 여러 사용자가 동시에 이용하면 각 사용자별로 별도 스레드가 생깁니다. 

스레드를 만들고, 그 안에 사용자 메시지를 추가한 뒤, 어시스턴트의 답변을 요청하는 일련의 과정을 코드로 수행해봅시다:


In [18]:
# 새로운 대화 스레드 생성 : 기억담당 (처음에는 메시지 없음)
thread = client.beta.threads.create()
# print('llm에 요청할 때 필요한 id :', thread.id)

# 4. 스레드에 사용자 프롬프트 추가
user_message = client.beta.threads.messages.create(
    thread_id = thread.id,
    role = "user",
    content = "오늘 날씨 몇도까지 올라가요?", # 사용자 프롬프트
)

user_message

Message(id='msg_IYje5UrH3rRxg6fzuvPCG2tg', assistant_id=None, attachments=[], completed_at=None, content=[TextContentBlock(text=Text(annotations=[], value='오늘 날씨 몇도까지 올라가요?'), type='text')], created_at=1751248307, incomplete_at=None, incomplete_details=None, metadata={}, object='thread.message', role='user', run_id=None, status=None, thread_id='thread_b0RWCaXCHqcwKO48jLrXg8zO')